# Actividad 3 — Transformación y enriquecimiento de datos

**Curso:** Proyecto Integrado III - Analítica de Datos
**Dataset:** `t22000t/bike-sharing-tabular` (UCI Bike Sharing — Capital Bikeshare, Washington D. C., 2011–2012)
**Etapa:** segunda del proyecto integrador (AE3) · **Grupo:** AE_Grupo 3

## Necesidad que se atiende

El área de operaciones de un sistema de bicicletas públicas necesita saber **cómo se distribuye la demanda
a lo largo del día y cómo cambia según el tipo de día, la temporada y la condición climática**, distinguiendo
el comportamiento de los usuarios ocasionales del de los suscriptores. El dataset no trae esa lectura hecha:
hay que prepararla.

## Preguntas de negocio (las mismas de la AE2)

1. ¿Cuánta demanda hay y cómo se reparte en el día?
2. ¿En qué franjas horarias se concentra?
3. ¿Cambia según el tipo de día (laborable / no laborable)?
4. ¿Cambia con la temporada y entre 2011 y 2012?
5. ¿Cambia con la condición climática y la temperatura?
6. ¿Qué peso tienen los usuarios casuales frente a los registrados?

## Qué hace este notebook

1. Carga el dataset y conserva una copia sin modificar.
2. Aplica y **justifica** las transformaciones de calidad, mostrando cuántos registros afecta cada una.
3. Incorpora variables calculadas y cruza la tabla diaria, comprobando que la unión no duplica registros.
4. Responde cada pregunta con una tabla y exporta los resultados a `.csv`.
5. Genera el `.txt` descriptivo del DataFrame final, por código.

El proyecto continúa con **el mismo dataset, las mismas preguntas y el mismo repositorio** de la AE2: no se
ajusta el alcance. Cualquier límite que aparezca se declara en las conclusiones.

> El análisis es **descriptivo**: describe lo observado en 2011–2012. No atribuye causas ni predice.

## 0. Entorno y rutas

El notebook se ejecuta desde `AE3/notebooks/`. Se sube por los directorios padres hasta encontrar
`src/clases_po` para importar las clases del proyecto, y se definen rutas **relativas** de salida.

In [1]:
import sys
from pathlib import Path

# Subir hasta la raíz del proyecto (la carpeta que contiene src/clases_po).
raiz = Path.cwd()
while not (raiz / "src" / "clases_po").exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz / "src"))

import pandas as pd
from clases_po import HFDatasetLoader, TransformadorBikes, EnriquecedorBikes, Reportes

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RUTA_DATOS = Path("../datos")
RUTA_RESULTADOS = Path("../resultados")
RUTA_DATOS.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

def exportar(tabla: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Guarda una tabla en CSV (UTF-8 con BOM para que Excel abra las tildes)."""
    ruta = RUTA_RESULTADOS / nombre
    tabla.to_csv(ruta, encoding="utf-8-sig")
    print(f"Guardado: {ruta}  ({tabla.shape[0]} filas × {tabla.shape[1]} columnas)")
    return tabla

print("Raíz del proyecto:", raiz)
print("Resultados en:", (RUTA_RESULTADOS / "").resolve())

Raíz del proyecto: /home/andresosoa/Documentos/Universidad/2026_4/Proyecto_Integrador_III/PI3_2026Q4
Resultados en: /home/andresosoa/Documentos/Universidad/2026_4/Proyecto_Integrador_III/PI3_2026Q4/AE3/resultados


/home/andresosoa/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga del dataset y copia sin modificar

`HFDatasetLoader` devuelve el DataFrame completo (17.379 registros horarios). Se guarda una copia
`df_crudo` que **no se toca en todo el notebook**: es la referencia para comparar el antes y el después.

In [2]:
loader = HFDatasetLoader("t22000t/bike-sharing-tabular", split="train")
df_crudo = loader.load().copy()

print(f"Filas: {len(df_crudo):,}  |  Columnas: {df_crudo.shape[1]}")
df_crudo.head(3)

Filas: 17,379  |  Columnas: 17


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.29,0.81,0.00,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.27,0.80,0.00,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.27,0.80,0.00,5,27,32


## 2. Transformación y validación

Cada regla vive en un método de `TransformadorBikes` y registra **cuántos registros afecta**. Cuando un
control no encuentra problemas, también queda registrado con cero: un control que no falla es un resultado,
no un paso vacío.

In [3]:
transformador = TransformadorBikes(df_crudo)
df_transformado = transformador.procesar()

reglas = transformador.tabla_reglas()
exportar(reglas, "tabla_0_reglas_transformacion.csv")
reglas

Guardado: ../resultados/tabla_0_reglas_transformacion.csv  (8 filas × 3 columnas)


,regla,detalle,registros_afectados
0,Tipo de fecha,dteday pasó de str a datetime64[us]; se creó f...,17379
1,Escalas,"temp→°C, atemp→°C, hum→%, windspeed→km/h (4 co...",17379
2,Códigos,5 variables categóricas traducidas a etiquetas,17379
3,Tipos numéricos,conteos y códigos convertidos a entero; sin id...,0
4,Duplicados,0 horas repetidas en fecha_hora,0
5,Consistencia cnt,cnt = casual + registered en los 17.379 registros,0
6,Cobertura,17379 de 17544 horas (165 faltantes en 76 días...,0
7,Selección,se conservan las 17 columnas originales y los ...,0


### 2.1 Antes y después

La comparación de estructura se hace sobre el DataFrame original y el transformado: filas, columnas,
tipos de dato, nulos y duplicados. Las filas no cambian porque **no se elimina ni se imputa nada**.

In [4]:
comparacion = transformador.comparacion(df_transformado)
exportar(comparacion, "tabla_0b_antes_despues.csv")
comparacion

Guardado: ../resultados/tabla_0b_antes_despues.csv  (5 filas × 3 columnas)


,antes,despues,cambio
filas,17379,17379,0
columnas,17,28,11
tipos_de_dato,3,5,2
valores_nulos,0,0,0
filas_duplicadas,0,0,0


### 2.2 Las horas que faltan

El periodo completo exigiría 24 horas × 731 días = **17.544** registros; el dataset tiene **17.379**.
Faltan **165 horas** repartidas en **76 días incompletos**. No se rellenan: un valor inventado en una hora
sin registro alteraría los promedios y las participaciones. Se deja la marca `dia_completo` para poder
filtrar o advertir en el análisis.

In [5]:
horas_por_dia = df_transformado.groupby("dteday").size()

cobertura = pd.DataFrame({
    "días del periodo": [horas_por_dia.size],
    "días completos (24 h)": [int((horas_por_dia == 24).sum())],
    "días incompletos": [int((horas_por_dia < 24).sum())],
    "horas esperadas": [int(horas_por_dia.size * 24)],
    "horas registradas": [int(horas_por_dia.sum())],
    "horas faltantes": [int(horas_por_dia.size * 24 - horas_por_dia.sum())],
    "mínimo de horas en un día": [int(horas_por_dia.min())],
}).T.rename(columns={0: "valor"})
exportar(cobertura, "tabla_0c_cobertura.csv")

print("Los cinco días con menos horas registradas:")
display(horas_por_dia.sort_values().head(5).to_frame("horas_registradas"))
cobertura

Guardado: ../resultados/tabla_0c_cobertura.csv  (7 filas × 1 columnas)
Los cinco días con menos horas registradas:


,horas_registradas
dteday,
2012-10-29,1
2011-01-27,8
2012-10-30,11
2011-01-18,12
2011-01-26,16


,valor
días del periodo,731
días completos (24 h),655
días incompletos,76
horas esperadas,17544
horas registradas,17379
horas faltantes,165
mínimo de horas en un día,1


## 3. Enriquecimiento

Enriquecer es construir información nueva **a partir de la que ya existe**: dimensiones de calendario,
participación de cada tipo de usuario, rangos de temperatura y el contexto del día. No se inventan
registros ni se usan fuentes externas (ver la justificación al final del documento).

Cada variable nueva queda anotada con su fórmula y su utilidad. Los cortes de las franjas horarias son
**operativos** (agrupan las horas de entrada y salida laboral), no intervalos estadísticos: por eso se documentan
en la clase y no se deducen de los datos.

In [6]:
enriquecedor = EnriquecedorBikes(df_transformado)
df = enriquecedor.procesar()

print(f"DataFrame final: {len(df):,} filas × {df.shape[1]} columnas")
enriquecedor.diccionario()

DataFrame final: 17,379 filas × 40 columnas


,variable,formula,utilidad
0,mes,nombre del mes a partir de dteday,comparar estacionalidad
1,es_fin_de_semana,"weekday en {0, 6} (0=domingo, 6=sábado)",separar uso de fin de semana del laboral
2,franja_horaria,"rangos de hr: 0-5, 6-8, 9-11, 12-15, 16-19, 20-23",comparar la demanda a lo largo del día
3,es_hora_pico,"franja_horaria en {Pico mañana, Pico tarde}",identificar las horas de mayor presión operativa
4,pct_casual,casual / cnt × 100,describir el perfil ocasional
5,pct_registered,registered / cnt × 100,describir el perfil suscrito
6,rango_temperatura,"cortes de temp_c en 5, 15 y 25 °C",comparar la demanda por temperatura
7,clima_adverso,"weathersit en {3, 4} (lluvia, nieve o tormenta)",aislar el efecto del mal tiempo
8,total_dia,suma de cnt por dteday (clave de unión),contexto del día para cada hora
9,promedio_hora_dia,media de cnt de las horas registradas del día,comparar la hora con un día típico


### 3.1 El cruce con la tabla diaria

La tabla diaria se construye **con los mismos datos** (suma de `cnt` por fecha) y se une al detalle por la
clave `dteday`, que es única por día. La consigna pide comprobar que la unión no multiplique registros:
se valida con `validate="many_to_one"` y se comparan filas y total de rentas antes y después.

In [7]:
validacion = pd.Series(enriquecedor.validacion_union).to_frame("valor")
exportar(validacion, "tabla_0d_validacion_union.csv")

assert validacion.loc["filas_antes", "valor"] == validacion.loc["filas_despues", "valor"]
assert validacion.loc["total_rentas_antes", "valor"] == validacion.loc["total_rentas_despues", "valor"]
print("La unión no alteró filas ni rentas: la tabla diaria no duplica el detalle horario.")
validacion

Guardado: ../resultados/tabla_0d_validacion_union.csv  (6 filas × 1 columnas)
La unión no alteró filas ni rentas: la tabla diaria no duplica el detalle horario.


,valor
filas_antes,17379
filas_despues,17379
total_rentas_antes,3292679
total_rentas_despues,3292679
dias_en_tabla_diaria,731
sin_coincidencia,0


## 4. Resultados: una tabla por pregunta

Todas las tablas se exportan a `.csv` y se leen con las mismas cifras que muestra el notebook.

### Pregunta 1 y 2 · ¿Cuánta demanda hay y cómo se reparte en el día?

In [8]:
tabla_franjas = (
    df.groupby("franja_horaria")
      .agg(horas_registradas=("cnt", "size"),
           rentas=("cnt", "sum"),
           promedio_por_hora=("cnt", "mean"),
           mediana=("cnt", "median"))
      .assign(participacion_pct=lambda t: t.rentas / t.rentas.sum() * 100)
      .round(1)
      .sort_values("rentas", ascending=False)
)
tabla_franjas.loc["Total"] = [
    tabla_franjas.horas_registradas.sum(), tabla_franjas.rentas.sum(),
    round(df.cnt.mean(), 1), round(df.cnt.median(), 1), 100.0,
]
exportar(tabla_franjas, "tabla_1a_franjas_horarias.csv")
tabla_franjas

Guardado: ../resultados/tabla_1a_franjas_horarias.csv  (7 filas × 5 columnas)


,horas_registradas,rentas,promedio_por_hora,mediana,participacion_pct
franja_horaria,,,,,
Pico tarde,"2,916.00","1,101,169.00",377.60,361.00,33.40
Mediodía,"2,915.00","728,134.00",249.80,222.00,22.10
Pico mañana,"2,179.00","470,304.00",215.80,131.00,14.30
Noche,"2,912.00","449,548.00",154.40,136.00,13.70
Mañana,"2,181.00","437,015.00",200.40,184.00,13.30
Madrugada,"4,276.00","106,509.00",24.90,13.00,3.20
Total,"17,379.00","3,292,679.00",189.50,142.00,100.00


In [9]:
tabla_horas = (
    df.groupby("hr")
      .agg(rentas=("cnt", "sum"), promedio=("cnt", "mean"), mediana=("cnt", "median"))
      .assign(participacion_pct=lambda t: t.rentas / t.rentas.sum() * 100)
      .round(1)
      .sort_values("promedio", ascending=False)
)
tabla_horas.index.name = "hora_del_dia"
exportar(tabla_horas, "tabla_1b_por_hora.csv")
tabla_horas.head(8)

Guardado: ../resultados/tabla_1b_por_hora.csv  (24 filas × 4 columnas)


,rentas,promedio,mediana,participacion_pct
hora_del_dia,,,,
17,336860,461.50,475.00,10.20
18,309772,425.50,418.50,9.40
8,261001,359.00,385.00,7.90
16,227748,312.00,304.50,6.90
19,226789,311.50,309.50,6.90
13,184919,253.70,224.00,5.60
12,184414,253.30,229.00,5.60
15,183149,251.20,227.00,5.60


**Lectura:** el sistema registró **3.292.679 rentas** en 17.379 horas. La franja de **pico de la tarde
(16:00–19:59)** concentra el **33,4 %** de las rentas ocupando solo el 16,8 % de las horas, y la hora 17
es la de mayor promedio (**461,5 rentas/hora**). La madrugada (00:00–05:59) aporta el **3,2 %**, con la
hora 4 como el mínimo (**6,4 rentas/hora**): entre las 17 y las 4 hay una diferencia de **72 veces**.

### Pregunta 3 · ¿Cambia según el tipo de día?

In [10]:
tabla_tipo_dia = (
    df.groupby("tipo_dia")
      .agg(horas_registradas=("cnt", "size"), rentas=("cnt", "sum"),
           promedio=("cnt", "mean"), mediana=("cnt", "median"),
           casual=("casual", "sum"), registered=("registered", "sum"))
      .assign(participacion_pct=lambda t: t.rentas / t.rentas.sum() * 100,
              pct_casual=lambda t: t.casual / (t.casual + t.registered) * 100)
      .round(1)
      .sort_values("rentas", ascending=False)
)
exportar(tabla_tipo_dia, "tabla_2_tipo_dia.csv")
tabla_tipo_dia

Guardado: ../resultados/tabla_2_tipo_dia.csv  (2 filas × 8 columnas)


,horas_registradas,rentas,promedio,mediana,casual,registered,participacion_pct,pct_casual
tipo_dia,,,,,,,,
Laborable,11865,2292410,193.20,151.00,303285,1989125,69.60,13.20
No laborable,5514,1000269,181.40,119.00,316732,683537,30.40,31.70


**Lectura:** los días laborables aportan el **69,6 %** de las rentas y un promedio de **193,2 rentas/hora**,
frente a **181,4** en los no laborables. La diferencia de promedios es pequeña, pero la **mediana** sí
cambia: 151 en laborables y 119 en no laborables. Es decir, el día no laborable tiene muchas horas de baja
demanda y unas pocas muy altas (fines de semana de buen clima), por eso su promedio se acerca al laborable.

### Pregunta 4 · ¿Cambia con la temporada y entre 2011 y 2012?

In [11]:
tabla_temporada = (
    df.groupby("temporada")
      .agg(horas_registradas=("cnt", "size"), rentas=("cnt", "sum"), promedio=("cnt", "mean"))
      .assign(participacion_pct=lambda t: t.rentas / t.rentas.sum() * 100)
      .round(1)
      .sort_values("promedio", ascending=False)
)
exportar(tabla_temporada, "tabla_3a_temporada.csv")

tabla_anio = (
    df.groupby("anio")
      .agg(horas_registradas=("cnt", "size"), rentas=("cnt", "sum"), promedio=("cnt", "mean"))
      .assign(participacion_pct=lambda t: t.rentas / t.rentas.sum() * 100)
      .round(1)
)
exportar(tabla_anio, "tabla_3b_anio.csv")

tabla_temporada_anio = (
    df.pivot_table(index="temporada", columns="anio", values="cnt", aggfunc="mean")
      .round(1)
      .sort_values(2012, ascending=False)
)
tabla_temporada_anio.columns = [f"promedio_{c}" for c in tabla_temporada_anio.columns]
exportar(tabla_temporada_anio, "tabla_3c_temporada_por_anio.csv")

display(tabla_temporada)
display(tabla_anio)
tabla_temporada_anio

Guardado: ../resultados/tabla_3a_temporada.csv  (4 filas × 4 columnas)
Guardado: ../resultados/tabla_3b_anio.csv  (2 filas × 4 columnas)
Guardado: ../resultados/tabla_3c_temporada_por_anio.csv  (4 filas × 2 columnas)


,horas_registradas,rentas,promedio,participacion_pct
temporada,,,,
Otoño,4496,1061129,236.00,32.20
Verano,4409,918589,208.30,27.90
Invierno,4232,841613,198.90,25.60
Primavera,4242,471348,111.10,14.30


,horas_registradas,rentas,promedio,participacion_pct
anio,,,,
2011,8645,1243103,143.80,37.80
2012,8734,2049576,234.70,62.20


,promedio_2011,promedio_2012
temporada,,
Otoño,187.30,284.30
Verano,157.70,259.00
Invierno,152.80,245.70
Primavera,72.50,147.80


**Lectura:** el otoño es la temporada de mayor demanda promedio (**236,0 rentas/hora**) y la primavera la
más baja (**111,1**), menos de la mitad. Entre años, 2012 promedia **234,7** frente a **143,8** en 2011
(**+63,2 %**). Esa diferencia mezcla estacionalidad con el crecimiento del propio sistema, así que no se
puede leer como un efecto del año: la comparación limpia requiere cruzar mes a mes, y se deja como
limitación para la etapa de métricas.

### Pregunta 5 · ¿Cambia con la condición climática y la temperatura?

In [12]:
tabla_clima = (
    df.groupby("clima")
      .agg(horas_registradas=("cnt", "size"), rentas=("cnt", "sum"),
           promedio=("cnt", "mean"), mediana=("cnt", "median"))
      .assign(participacion_pct=lambda t: t.rentas / t.rentas.sum() * 100)
      .round(1)
      .sort_values("promedio", ascending=False)
)
exportar(tabla_clima, "tabla_4a_clima.csv")

tabla_temperatura = (
    df.groupby("rango_temperatura", observed=True)
      .agg(horas_registradas=("cnt", "size"), rentas=("cnt", "sum"),
           promedio=("cnt", "mean"), mediana=("cnt", "median"))
      .round(1)
)
exportar(tabla_temperatura, "tabla_4b_rango_temperatura.csv")

display(tabla_clima)
tabla_temperatura

Guardado: ../resultados/tabla_4a_clima.csv  (4 filas × 5 columnas)


Guardado: ../resultados/tabla_4b_rango_temperatura.csv  (4 filas × 4 columnas)


,horas_registradas,rentas,promedio,mediana,participacion_pct
clima,,,,,
Despejado,11413,2338173,204.90,159.00,71.00
Neblina o nublado,4544,795952,175.20,133.00,24.20
Lluvia o nieve ligera,1419,158331,111.60,63.00,4.80
Lluvia fuerte o tormenta,3,223,74.30,36.00,0.00


,horas_registradas,rentas,promedio,mediana
rango_temperatura,,,,
Fría (≤5 °C),193,9388,48.60,25.00
Templada (5-15 °C),5248,555609,105.90,70.00
Cálida (15-25 °C),6103,1160028,190.10,150.00
Calurosa (>25 °C),5835,1567654,268.70,236.00


**Lectura:** con cielo despejado el promedio es **204,9 rentas/hora**; con lluvia o nieve ligera baja a
**111,6** (**−45,5 %**). El nivel de tormenta solo aparece en **3 horas** (74,3 de promedio): no alcanza
para concluir nada, y así se reporta. La temperatura muestra una relación monótona: de **48,6** en horas
frías (≤5 °C) a **268,7** por encima de 25 °C.

### Pregunta 6 · ¿Qué peso tienen los usuarios casuales y los registrados?

In [13]:
tabla_usuario = pd.DataFrame({
    "rentas": [df.casual.sum(), df.registered.sum(), df.cnt.sum()],
    "participacion_pct": [df.casual.sum() / df.cnt.sum() * 100,
                          df.registered.sum() / df.cnt.sum() * 100, 100.0],
    "promedio_por_hora": [df.casual.mean(), df.registered.mean(), df.cnt.mean()],
}, index=["Casual", "Registrado", "Total"]).round(1)
exportar(tabla_usuario, "tabla_5a_tipo_usuario.csv")

tabla_usuario_dia = (
    df.groupby("tipo_dia")[["casual", "registered"]].sum()
      .assign(total=lambda t: t.casual + t.registered,
              pct_casual=lambda t: t.casual / t.total * 100,
              pct_registered=lambda t: t.registered / t.total * 100)
      .round(1)
)
exportar(tabla_usuario_dia, "tabla_5b_usuario_por_tipo_dia.csv")
tabla_usuario_dia

Guardado: ../resultados/tabla_5a_tipo_usuario.csv  (3 filas × 3 columnas)


Guardado: ../resultados/tabla_5b_usuario_por_tipo_dia.csv  (2 filas × 5 columnas)


,casual,registered,total,pct_casual,pct_registered
tipo_dia,,,,,
Laborable,303285,1989125,2292410,13.20,86.80
No laborable,316732,683537,1000269,31.70,68.30


**Lectura:** los suscriptores son el **81,2 %** de las rentas (2.672.662) y los casuales el **18,8 %**
(620.017). El reparto cambia con el tipo de día: en laborables el casual es **13,2 %** y en no laborables
sube a **31,7 %**, casi el doble y medio. El perfil ocasional pesa más cuando no se trabaja.

### Concentración de la operación

In [14]:
concentracion = df.pct_hora_del_dia.describe().round(2).to_frame("pct_hora_del_dia")
exportar(concentracion, "tabla_6a_concentracion_horaria.csv")

top_dias = (
    df.groupby("dteday")
      .agg(rentas=("cnt", "sum"), horas_registradas=("cnt", "size"))
      .sort_values("rentas", ascending=False)
      .head(5)
      .round(1)
)
top_dias.index = top_dias.index.strftime("%Y-%m-%d")
exportar(top_dias, "tabla_6b_dias_mayor_demanda.csv")

display(concentracion)
top_dias

Guardado: ../resultados/tabla_6a_concentracion_horaria.csv  (8 filas × 1 columnas)
Guardado: ../resultados/tabla_6b_dias_mayor_demanda.csv  (5 filas × 2 columnas)


,pct_hora_del_dia
count,"17,379.00"
mean,4.21
std,3.47
min,0.01
25%,1.24
50%,3.66
75%,6.34
max,100.00


,rentas,horas_registradas
dteday,,
2012-09-15,8714,24
2012-09-29,8555,24
2012-09-22,8395,24
2012-03-23,8362,24
2012-05-19,8294,24


**Lectura:** la hora mediana representa el **3,66 %** de la renta de su día, pero hay días con una
sola hora registrada en los que esa hora es el 100 %. Por eso la participación horaria se interpreta junto
con `horas_registradas_dia`. Los tres primeros días de mayor demanda son de septiembre de 2012 (15, 22 y 29), con más de
8.300 rentas diarias.

## 5. Exportaciones

Se guardan, por código y en rutas relativas:

- `datos/dataset_preparado.csv` — el DataFrame final completo.
- `resultados/tabla_*.csv` — una tabla por pregunta (y las de control).
- `resultados/resumen_dataframe.txt` — el reporte descriptivo del DataFrame final.

El `.txt` se genera con la clase `Reportes`, en el mismo formato trabajado en Triple A, y se **regenera en
cada ejecución**: su contenido sale del DataFrame que se acaba de construir, no de una versión anterior.

In [15]:
ruta_dataset = RUTA_DATOS / "dataset_preparado.csv"
df.to_csv(ruta_dataset, index=False, encoding="utf-8-sig")
print(f"Dataset preparado: {ruta_dataset}  ({len(df):,} filas × {df.shape[1]} columnas)")

reportes = Reportes()
ruta_txt = reportes.guardar_reporte(df, RUTA_RESULTADOS / "resumen_dataframe.txt")
print(f"Reporte descriptivo: {ruta_txt}")
print("-" * 60)
print("".join(ruta_txt.read_text(encoding="utf-8").splitlines(keepends=True)[:8]))

Dataset preparado: ../datos/dataset_preparado.csv  (17,379 filas × 40 columnas)
Reporte descriptivo: ../resultados/resumen_dataframe.txt
------------------------------------------------------------
REPORTE DEL DATAFRAME
Filas: 17379
Columnas: 40
Duplicados: 0

Columnas y tipos:
- instant: int64



### 5.1 Verificación de las salidas

Se comprueba que los archivos existan, que el `.txt` corresponda al DataFrame final (filas y columnas) y
que los totales y porcentajes sean consistentes entre sí.

In [16]:
texto = ruta_txt.read_text(encoding="utf-8")

comprobaciones = {
    "filas del notebook": len(df),
    "filas declaradas en el .txt": int(texto.split("Filas: ")[1].split("\n")[0]),
    "columnas del notebook": df.shape[1],
    "columnas declaradas en el .txt": int(texto.split("Columnas: ")[1].split("\n")[0]),
    "suma cnt original": int(df_crudo.cnt.sum()),
    "suma cnt final": int(df.cnt.sum()),
    "suma de participación por franja (%)": round(tabla_franjas.loc["Total", "participacion_pct"], 1),
    "suma de participación por usuario (%)": round(tabla_usuario.loc["Total", "participacion_pct"], 1),
    "archivos de resultados": len(list(RUTA_RESULTADOS.glob("*"))),
}
pd.Series(comprobaciones).to_frame("valor")

,valor
filas del notebook,"17,379.00"
filas declaradas en el .txt,"17,379.00"
columnas del notebook,40.00
columnas declaradas en el .txt,40.00
suma cnt original,"3,292,679.00"
suma cnt final,"3,292,679.00"
suma de participación por franja (%),100.00
suma de participación por usuario (%),100.00
archivos de resultados,17.00


In [17]:
assert comprobaciones["filas del notebook"] == comprobaciones["filas declaradas en el .txt"]
assert comprobaciones["columnas del notebook"] == comprobaciones["columnas declaradas en el .txt"]
assert comprobaciones["suma cnt original"] == comprobaciones["suma cnt final"]
assert comprobaciones["suma de participación por franja (%)"] == 100.0
assert comprobaciones["suma de participación por usuario (%)"] == 100.0
print("Todas las comprobaciones pasaron.")

Todas las comprobaciones pasaron.


## 6. Conclusiones del avance

- El dataset quedó **preparado**: 8 reglas de transformación documentadas con su efecto (0 registros
  eliminados) y **12 variables nuevas**; el DataFrame final tiene **40 columnas** y las mismas **17.379**
  filas del original.
- La demanda **no se reparte de forma uniforme**: el pico de la tarde concentra un tercio de las rentas y la
  madrugada apenas el 3 %. Esa concentración es la que justifica dimensionar la operación por franjas.
- El **tipo de día** cambia la composición más que el promedio: en días no laborables el usuario casual
  duplica su peso (31,7 % frente a 13,2 %).
- El **clima** sí separa los niveles de demanda (despejado 204,9 vs. lluvia 111,6), mientras que el nivel de
  tormenta no tiene registros suficientes para afirmar nada.
- La **comparación entre años** no es concluyente por sí sola: el sistema creció entre 2011 y 2012 y eso se
  mezcla con la estacionalidad.

### Limitaciones que se mantienen

- La unidad es la **hora del sistema completo**: no hay estaciones, recorridos ni usuarios individuales.
- Faltan **165 horas** en 76 días; no se imputaron y quedan marcadas.
- No hay información de flota, capacidad ni mantenimiento: no se puede describir la oferta ni la rentabilidad.
- La condición de tormenta tiene **3 registros**, insuficientes para un promedio representativo.

### Qué sigue (AE4)

Estas tablas son la base de la visualización: la franja horaria, el tipo de día, la temporada y el clima ya
quedaron calculados y exportados, de modo que las gráficas se construyan sobre los mismos números.

## 7. Reproducibilidad

```bash
cd AE3/notebooks
jupyter nbconvert --to notebook --execute --inplace EA3_transformacion_enriquecimiento.ipynb
```

Dependencias: `pandas`, `datasets`, `nbconvert` (ver `requirements.txt`). Las rutas de salida son relativas
a la carpeta del notebook, y los archivos se regeneran en cada ejecución.